<a href="https://colab.research.google.com/github/adminsanjay/ML-projects/blob/main/vgg16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================
# VGG16 AUDIO CLASSIFIER
# ================================

import kagglehub
path = kagglehub.dataset_download("mohammedabdeldayem/the-fake-or-real-dataset")
print("Dataset Path:", path)

import os
import librosa
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Input
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import accuracy_score, classification_report

# ================================
# DATA GENERATOR
# ================================
class AudioDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, labels, batch_size=32, sr=16000, n_mfcc=40, target_frames=64):
        self.file_paths = file_paths
        self.labels = labels
        self.batch_size = batch_size
        self.sr = sr
        self.n_mfcc = n_mfcc
        self.target_frames = target_frames
        self.indices = np.arange(len(self.file_paths))

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        batch_paths = [self.file_paths[i] for i in batch_indices]
        batch_labels = [self.labels[i] for i in batch_indices]

        X = np.empty((len(batch_paths), self.n_mfcc, self.target_frames, 1))
        y = np.empty((len(batch_paths), 2))

        for i, path in enumerate(batch_paths):
            try:
                audio, _ = librosa.load(path, sr=self.sr)

                if len(audio) < self.sr:
                    audio = np.pad(audio, (0, self.sr - len(audio)))

                mfcc = librosa.feature.mfcc(y=audio, sr=self.sr, n_mfcc=self.n_mfcc)
                mfcc = (mfcc - np.mean(mfcc)) / (np.std(mfcc) + 1e-8)

                if mfcc.shape[1] < self.target_frames:
                    mfcc = np.pad(mfcc, ((0, 0), (0, self.target_frames - mfcc.shape[1])))
                else:
                    mfcc = mfcc[:, :self.target_frames]

                X[i] = np.expand_dims(mfcc, axis=-1)

            except:
                X[i] = np.zeros((self.n_mfcc, self.target_frames, 1))

            y[i] = tf.keras.utils.to_categorical(batch_labels[i], 2)

        return X, y

# ================================
# LOAD PATHS
# ================================
def get_paths_and_labels(base_path):
    paths, labels = [], []

    for label_name in ['real', 'fake']:
        label_dir = os.path.join(base_path, label_name)
        if not os.path.exists(label_dir):
            continue

        label = 0 if label_name == 'real' else 1

        for file in os.listdir(label_dir):
            paths.append(os.path.join(label_dir, file))
            labels.append(label)

    return paths, labels

# ================================
# VGG16 MODEL
# ================================
def create_vgg16_model(input_shape):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

    for layer in base_model.layers:
        layer.trainable = False

    inputs = Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(3, (1,1), padding='same')(inputs)
    x = tf.keras.layers.Resizing(64,64)(x)

    x = base_model(x)
    x = Flatten()(x)

    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation='relu')(x)

    outputs = Dense(2, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# ================================
# MAIN
# ================================
dataset_dir = "/kaggle/input/the-fake-or-real-dataset/for-rerec/for-rerecorded"

train_paths, train_labels = get_paths_and_labels(os.path.join(dataset_dir, 'training'))
val_paths, val_labels = get_paths_and_labels(os.path.join(dataset_dir, 'validation'))
test_paths, test_labels = get_paths_and_labels(os.path.join(dataset_dir, 'testing'))

train_gen = AudioDataGenerator(train_paths, train_labels)
val_gen = AudioDataGenerator(val_paths, val_labels)
test_gen = AudioDataGenerator(test_paths, test_labels)

model = create_vgg16_model((40,64,1))

checkpoint = ModelCheckpoint("vgg16_model.keras", save_best_only=True, monitor='val_accuracy')
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=[checkpoint, early_stop])

# Evaluation
pred_probs = model.predict(test_gen)
pred = np.argmax(pred_probs, axis=1)

print("Accuracy:", accuracy_score(test_labels, pred))
print(classification_report(test_labels, pred))

Using Colab cache for faster access to the 'the-fake-or-real-dataset' dataset.
Dataset Path: /kaggle/input/the-fake-or-real-dataset
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.6439 - loss: 0.9338

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


319/319 ━━━━━━━━━━━━━━━━━━━━ 212s 633ms/step - accuracy: 0.7172 - loss: 0.6352 - val_accuracy: 0.5022 - val_loss: 0.9960
Epoch 2/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 108s 337ms/step - accuracy: 0.8040 - loss: 0.4314 - val_accuracy: 0.8365 - val_loss: 0.3765
Epoch 3/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 108s 337ms/step - accuracy: 0.8306 - loss: 0.3690 - val_accuracy: 0.8017 - val_loss: 0.4629
Epoch 4/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 140s 331ms/step - accuracy: 0.8438 - loss: 0.3432 - val_accuracy: 0.8525 - val_loss: 0.3348
Epoch 5/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 104s 327ms/step - accuracy: 0.8568 - loss: 0.3134 - val_accuracy: 0.8663 - val_loss: 0.3107
Epoch 6/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 104s 326ms/step - accuracy: 0.8640 - loss: 0.2996 - val_accuracy: 0.7638 - val_loss: 0.4600
Epoch 7/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 107s 334ms/step - accuracy: 0.8695 - loss: 0.2862 - val_accuracy: 0.8810 - val_loss: 0.2836
Epoch 8/20
319/319 ━━━━━━━━━━━━━━━━━━━━ 104s 326ms/step - accuracy: 0.8744 - loss: 0.27